# Post-processing and standard plots

This tutorial introduces the standardized post-processing interface. We run a small Vlasov–Ampère example, turn its raw output into labeled arrays with `sim.pproc(load=True)`, and make the plots most commonly used to inspect a simulation.

For a production run you can skip the simulation setup and construct both objects with `path_out="path/to/sim"` instead.

In [ ]:
import os
import tempfile

from struphy import (
    BinningPlot,
    BoundaryParameters,
    DerhamOptions,
    EnvironmentOptions,
    LoadingParameters,
    SavingParameters,
    Simulation,
    SortingParameters,
    Time,
    WeightsParameters,
    domains,
    grids,
    maxwellians,
    perturbations,
)
from struphy.models import VlasovAmpereOneSpecies

## Create a compact demonstration run

Post-processing operates on a completed run. The small setup below saves an electric field, a few marker trajectories, scalar diagnostics, and a binned $(\eta_1,v_1)$ distribution. These are the main output types handled by the plotting interface.

In [ ]:
model = VlasovAmpereOneSpecies(alpha=1.0, epsilon=-1.0, with_B0=False)
model.em_fields.e_field.save_data = True
model.em_fields.phi.save_data = True
model.kinetic_ions.var.save_data = True

model.propagators.push_eta.options = model.propagators.push_eta.Options()
model.propagators.coupling_va.options = model.propagators.coupling_va.Options()
model.initial_poisson.options = model.initial_poisson.Options(stab_mat="M0")

binplot = BinningPlot(
    slice="e1_v1",
    n_bins=(32, 32),
    ranges=((0.0, 1.0), (-5.0, 5.0)),
)
model.kinetic_ions.set_markers(
    loading_params=LoadingParameters(ppc=32, seed=1234),
    weights_params=WeightsParameters(control_variate=True),
    boundary_params=BoundaryParameters(),
    sorting_params=SortingParameters(boxes_per_dim=(4, 1, 1), do_sort=True),
    saving_params=SavingParameters(n_markers=12, binning_plots=(binplot,)),
)

background = maxwellians.Maxwellian3D(n=(1.0, None))
model.kinetic_ions.var.add_background(background)
density_mode = perturbations.ModesCos(ls=(1,), amps=(1e-3,))
model.kinetic_ions.var.add_initial_condition(maxwellians.Maxwellian3D(n=(1.0, density_mode)))

In [ ]:
demo_tmp = tempfile.TemporaryDirectory(prefix="struphy_postprocessing_")
demo_root = demo_tmp.name

env = EnvironmentOptions(
    out_folders=demo_root,
    sim_folder="vlasov_ampere_demo",
    save_restart=False,
)
sim = Simulation(
    model=model,
    env=env,
    time_opts=Time(dt=0.1, Tend=0.4),
    domain=domains.Cuboid(r1=2 * 3.141592653589793),
    grid=grids.TensorProductGrid(num_elements=(8, 1, 1)),
    derham_opts=DerhamOptions(degree=(2, 1, 1)),
)
sim.run()
print(f"Raw output: {sim.env.path_out}")

## Process and load the output

`sim.pproc(load=True)` evaluates saved FEEC fields, organizes particle diagnostics, and returns the loaded plotting data. `physical=True` additionally creates physical field components; `create_vtk=False` keeps this notebook quick. With `force=False`, an existing post-processing directory is reused.

The result exposes the output as `StruphyArray` objects. Each array carries named dimensions, coordinates, units, and a display label. The lower-level `PostProcessor` and `PlottingData` classes remain available when processing and loading should be performed separately.

In [ ]:
pdata = sim.pproc(
    physical=True,
    create_vtk=False,
    force=False,
    load=True,
)

The containers are discoverable, so a plotting script does not need to guess what a run saved. Dictionary-style and attribute-style access are both supported.

In [ ]:
print("scalars:", pdata.scalars.keys())
print("field species:", pdata.spline_values.keys())
print("kinetic species:", pdata.f.keys())
print("particle orbits:", pdata.orbits.keys())

phase_space = pdata.f.kinetic_ions["e1_v1_density"]["f_binned"]
print(phase_space)
print("dimensions:", phase_space.dims)
print("time coordinate:", phase_space.coord("t"))

## Scalar overview and time series

`pdata.plot.scalars()` gives a quick overview of every recorded scalar. If `total_energy` is available, it is also used for the conservation-error panel. Individual time series can be shown on linear or logarithmic axes, and an exponential fit can be restricted to a chosen time interval.

In [ ]:
pdata.plot.scalars(
    error_panel="total_energy",
)

In [ ]:
electric_energy = pdata.scalars["electric_energy"]
energy_plot = pdata.plot.time_series(
    "electric_energy",
    logy=True,
    fit=True,
    fit_window=(0.0, 0.4 * pdata.units.t),
    fit_of_sqrt=True,  # report the field-amplitude rate of this quadratic energy
    title="Electric-field energy",
)
print("fit result (gamma, intercept, index window):", energy_plot.fit_results[0])

## Two-dimensional data

Named selection keeps plots readable. Use `isel` for an integer index and `at` for the point nearest a coordinate value. A single phase-space snapshot can then be passed directly to `pdata.plot.slice()`; coordinates and labels come from the array.

In [ ]:
final_distribution = phase_space.isel(t=-1)
pdata.plot.slice(
    final_distribution,
    equal_aspect=False,
    title="Final phase-space distribution",
)

For a compact view of the evolution, `pdata.plot.panels()` chooses evenly spaced snapshots. `shared_clim=True` makes their colors directly comparable.

In [ ]:
pdata.plot.panels(
    phase_space,
    nrows=1,
    ncols=5,
    shared_clim=True,
    equal_aspect=False,
    title="Phase-space evolution",
)

## Interactive plots

`pdata.plot.slider()` adds a time slider to any `(t, a, b)` array. In JupyterLab, run `%matplotlib widget` before this cell if `ipympl` is installed; the default inline backend still displays the initial frame. Keep the returned object alive so its widget callbacks remain connected.

In [ ]:
phase_slider = pdata.plot.slider(
    phase_space,
    equal_aspect=False,
    title="Phase-space distribution",
)

Saved marker orbits use a three-dimensional trajectory plot with a time slider. `max_markers` limits rendering cost for large production runs.

In [ ]:
orbit_plot = pdata.plot.orbits(
    "kinetic_ions",
    max_markers=12,
    show_paths=True,
)

## Save standard output

The same plot objects support `.save(path)`. For a complete scalar report, `save_scalar_plots()` writes a CSV table, an overview, and one PNG per scalar beneath `post_processing/scalars/`.

In [ ]:
written = pdata.save_scalar_plots()
print("Wrote:")
for path in written:
    print(" ", os.path.relpath(path, pdata.path_out))

## Apply the workflow to another run

For an already completed simulation, the complete loading pattern is:

```python
path_out = "/path/to/sim_1"
from struphy import post_process
pdata = post_process(path_out=path_out, physical=True, force=False)
```

Use `pdata.scalars`, `pdata.spline_values`, `pdata.f`, `pdata.orbits`, and `pdata.n_sph` to discover and plot the data available in that run.